# Lecture 11 — Randomness, Iteration, and Simulation

**데이터과학입문 · 2026-1 · 부산대학교 정보컴퓨터공학부**  
**Donghee Choi** · `dchoi@pusan.ac.kr`

이 노트북은 강의 슬라이드 `11_randomness_simulation.pdf`의 모든 코드를 그대로 실행할 수 있도록 정리한 것입니다.
Colab에서 위쪽 *Open in Colab* 버튼을 눌러 바로 실행할 수 있습니다.

## 학습 목표
1. **Randomness** — `numpy`의 `random.choice`로 무작위 표본을 만들고, Boolean 비교로 결과를 세는 방법.
2. **Iteration** — `for` 반복으로 같은 실험을 여러 번 실행하고 결과를 누적하는 패턴.
3. **Simulation** — 한 번의 실험 = 한 개의 통계량, 많은 반복 = 통계량의 *경험적 분포*.
4. **사례 연구** — Monty Hall 문제를 시뮬레이션으로 해결.

**선수 지식**: Lecture 7 (Python/NumPy), 8 (Pandas), 10 (Sampling 개념)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## Part A — Randomness in Code

작은 numpy primitive 만으로 무작위 결과를 다룬다. 먼저 비교(comparison) → 집계(aggregating) → 무작위 추출(`np.random.choice`) 순서로.

### A.1 Comparison Expressions — 비교식의 결과는 `bool`

비교 연산자(`>`, `<`, `==`, `!=`, `>=`, `<=`)는 항상 `True` 또는 `False` 를 반환한다.

In [ ]:
x = 2; y = 3
print(x > 1)        # True
print(x > y)        # False
print(y >= 3)       # True
print(x == y)       # False
print(x != 2)       # False
print(2 < x < 5)    # False  (chained: 2 < x AND x < 5)

**문자열 비교** — 알파벳 순(앞글자부터 사전식), 같은 prefix 면 짧은 쪽이 작다.

In [ ]:
print('Dog' > 'Catastrophe' > 'Cat')   # True
print('a' < 'b')                       # True   (alphabetical)
print('abc' < 'abcd')                  # True   (shorter < longer when prefix matches)

### A.2 Aggregating Comparisons — bool 들의 합 = True 의 개수

`True == 1`, `False == 0` 이므로, bool list/array 를 `sum` 하면 **True 의 개수가 나온다.**

In [ ]:
print(1 + 0 + 1)                  # 2
print(True + False + True)        # 2  (True==1, False==0)

print(sum([1, 0, 1]))             # 2
print(sum([True, False, True]))   # 2

### A.3 Comparing an Array and a Value — 배열에 비교 적용 + 집계

- **Comparison applies to each element** of the array → result is a Boolean array.
- **The result array can be aggregated** the same way as the scalar case.

이 두 가지가 오늘 모든 시뮬레이션의 핵심 엔진.

In [ ]:
tosses = np.array(['Tails', 'Heads', 'Tails', 'Heads', 'Heads'])
tosses == 'Tails'
# array([ True, False,  True, False, False])

In [ ]:
np.count_nonzero(tosses == 'Tails')   # 2

In [ ]:
# True == 1 이므로 sum 도 같은 값
sum(tosses == 'Tails')                # 2

### A.4 `np.random.choice` — drawing one outcome

출력 가능한 값을 array 로 만들고, `np.random.choice` 로 그 중 하나를 무작위로 뽑는다. 셀을 여러 번 실행해 보면 결과가 매번 달라짐.

In [ ]:
two_groups = np.array(['treatment', 'control'])
np.random.choice(two_groups)

### A.5 `np.random.choice` — drawing many outcomes

두 번째 인자에 정수 `N` 을 넘기면 `N` 개를 한꺼번에 뽑는다. 결과는 numpy array.

In [ ]:
np.random.choice(two_groups, 10)

### A.6 Sampling **With** vs **Without** Replacement

- **With replacement** (default, `replace=True`) — each draw is **independent**; same item can appear multiple times. 동전 던지기·주사위 굴리기·반복 설문이 이쪽.
- **Without replacement** (`replace=False`) — draws are **linked**, each item appears at most once. 카드 분배·위원회 추첨·서로 다른 사람 설문이 이쪽. 최대 표본 수 = 배열 길이.

독립성이 시뮬레이션을 단순하게 만들어주기 때문에 오늘 대부분 default(with replacement) 를 사용한다.

In [ ]:
deck = np.array(['A', 'B', 'C', 'D', 'E'])

# WITH replacement — 같은 item 이 중복될 수 있음
np.random.choice(deck, 4)
# 예: array(['C', 'A', 'C', 'B'])

In [ ]:
# WITHOUT replacement — 모두 distinct
np.random.choice(deck, 4, replace=False)
# 예: array(['D', 'A', 'E', 'B'])

In [ ]:
# replace=False 면 배열 길이보다 많이 뽑을 수 없음 (아래는 ValueError)
# np.random.choice(deck, 6, replace=False)

---
## Part B — Iteration

같은 실험을 컴퓨터에게 여러 번 반복시키는 방법.

### B.1 `for`와 `np.arange`

In [ ]:
for i in np.arange(5):
    print('iteration', i)

### B.2 `np.append`로 결과 누적

**핵심 패턴**: 빈 array → 루프 → 매번 append → 끝나면 결과 array.

In [ ]:
outcomes = np.array([])
for i in np.arange(5):
    pick = np.random.choice(['H', 'T'])
    outcomes = np.append(outcomes, pick)
outcomes

### B.3 예제 — 주사위 베팅

**규칙**: 주사위를 던져 1, 2가 나오면 \$1 손실 / 3, 4면 본전 / 5, 6이면 \$1 이득.

In [ ]:
def one_bet(x):
    """주사위 눈 x에 대한 한 번의 베팅 결과 (-1 / 0 / +1)"""
    if x <= 2:
        return -1
    elif x <= 4:
        return 0
    else:
        return 1

def bet_on_one_roll():
    """주사위 한 번 굴려 베팅 결과 반환"""
    x = np.random.choice(np.arange(1, 7))
    return one_bet(x)

bet_on_one_roll()

### B.4 300번 반복 + 분포 시각화

In [ ]:
outcomes = np.array([])
for i in np.arange(300):
    outcomes = np.append(outcomes, bet_on_one_roll())

table = pd.DataFrame({'Outcome': outcomes})
counts = (table.groupby('Outcome')['Outcome']
               .count()
               .reset_index(name='count'))
counts

In [ ]:
ax = counts.plot.barh(x='Outcome', y='count', legend=False)
ax.set_xlabel('count'); ax.set_title('300 dice bets')
plt.show()

각 결과(-1, 0, +1)는 약 1/3 비율로 나옵니다. 이것이 베팅의 *경험적 분포*.

---
## Part C — Simulation

한 trial이 *숫자*를 만들 때, 이를 많이 반복해서 그 숫자의 분포를 그립니다.

### C.1 한 번의 시뮬레이션 값

**실험**: 동전 100번 던지고, 그중 Heads의 개수를 세기.

In [ ]:
coin = np.array(['Heads', 'Tails'])

def one_simulated_value():
    flips = np.random.choice(coin, 100)
    return np.count_nonzero(flips == 'Heads')

one_simulated_value()

### C.2 20,000번 반복 → 분포 만들기

In [ ]:
num_repetitions = 20000

heads = np.array([])
for i in np.arange(num_repetitions):
    heads = np.append(heads, one_simulated_value())

len(heads)

In [ ]:
rst = pd.DataFrame({'Number of Heads': heads})
ax = rst['Number of Heads'].plot.hist(bins=np.arange(30, 71), edgecolor='white')
ax.set_xlabel('Number of Heads in 100 flips')
ax.set_title(f'Empirical distribution over {num_repetitions} repetitions')
plt.show()

**관찰**: 히스토그램이 50 근처에 몰려 있고 종 모양 (Normal distribution의 전조).  
시뮬레이션이 확률에 대한 사실을 *수식 없이* 발견했습니다.

### C.3 (보조) — N번 던질 때 Heads가 적어도 한 번 나올 확률

이 경우는 시뮬레이션보다 수식이 더 쉽습니다 — 알맞은 도구를 고르는 것도 중요.

In [ ]:
tosses = np.arange(1, 51)
results = pd.DataFrame({
    'Tosses': tosses,
    'Chance of at least one H': 1 - (1/2)**tosses
})
ax = results.plot.scatter(x='Tosses', y='Chance of at least one H')
plt.show()

---
## Part D — Monty Hall Problem

**규칙**: 문 3개. 하나 뒤에는 자동차, 둘 뒤에는 염소. 당신이 한 문을 고르면 진행자가 *다른* 문 중에서 *염소가 있는* 문을 열어 보여줍니다. 바꿀까요?

In [ ]:
goats = np.array(['first goat', 'second goat'])
hidden = np.append(goats, 'car')

def other_goat(x):
    if x == 'first goat':
        return 'second goat'
    if x == 'second goat':
        return 'first goat'

def monty_hall_game():
    """한 게임 결과: [내가 고른 것, 진행자가 보여준 것, 남은 문]"""
    contestant = np.random.choice(hidden)
    if contestant == 'car':
        revealed = np.random.choice(goats)
        remaining = other_goat(revealed)
    elif contestant == 'first goat':
        revealed = 'second goat'
        remaining = 'car'
    else:  # 'second goat'
        revealed = 'first goat'
        remaining = 'car'
    return [contestant, revealed, remaining]

monty_hall_game()

### D.1 10,000 게임 시뮬레이션

In [ ]:
games = pd.DataFrame(columns=['Guess', 'Revealed', 'Remaining'])
for i in np.arange(10000):
    games.loc[i] = monty_hall_game()

games.head()

In [ ]:
# 처음 고른 항목 분포 (= '바꾸지 않을 때 결과')
original = (games.groupby('Guess')['Guess']
                  .count()
                  .reset_index(name='if_kept'))

# 남은 문 분포 (= '바꿀 때 결과')
remaining = (games.groupby('Remaining')['Remaining']
                  .count()
                  .reset_index(name='if_switched'))

joined = original.merge(remaining, left_on='Guess', right_on='Remaining')
joined

In [ ]:
ax = joined.set_index('Guess')[['if_kept', 'if_switched']].plot.barh()
ax.set_xlabel('count'); ax.set_title('Monty Hall — keep vs switch (10,000 games)')
plt.show()

**관찰**: `car`가 *남은 문*에 나오는 비율이 약 67% — 즉 **switch가 2/3 확률로 승리**.

직관이 잘못된 경우, 시뮬레이션이 가장 빠른 증명입니다.

---
## 정리 — Simulation Recipe

1. **실험 정의** — 한 번의 trial을 함수로 만들고, 관심 통계량을 반환.
2. **반복** — `for i in np.arange(N)`.
3. **누적** — 빈 array에 `np.append`.
4. **시각화** — `plot.hist` / `plot.barh`.
5. **읽기** — empirical distribution.

다음 시간에는 이 패턴을 데이터(주사위/동전 대신 실제 표본)에 적용하여 **Estimation, Bootstrap, Confidence Interval**을 다룹니다.